# GAN and VAE Practical Demo

This notebook implements the requested Fashion-MNIST GAN and MNIST Autoencoder/VAE practicals.

## Part 1 — GAN using Fashion-MNIST

**Goal:** Train a GAN for 25 epochs, save 4×4 generated image grids every 5 epochs, record Generator/Discriminator losses, and visualize training progress.

In [ ]:

import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)

BATCH_SIZE = 128
EPOCHS = 25
NOISE_DIM = 100
IMAGE_SIZE = 28
CHANNELS = 1
SAVE_EVERY = 5

os.makedirs("gan_generated", exist_ok=True)

print("TensorFlow version:", tf.__version__)


### Hardcoded noise vector
A fixed noise vector is used so that the same latent inputs can be compared across epochs.

In [ ]:

hardcoded_noise = tf.random.normal([16, NOISE_DIM], seed=42)
print("Hardcoded noise shape:", hardcoded_noise.shape)


In [ ]:

# Load Fashion-MNIST
(x_train, _), (_, _) = tf.keras.datasets.fashion_mnist.load_data()

# Normalize pixel values from [0,255] to [-1,1]
x_train = (x_train.astype("float32") - 127.5) / 127.5
x_train = np.expand_dims(x_train, axis=-1)

train_dataset = (
    tf.data.Dataset.from_tensor_slices(x_train)
    .shuffle(60000)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE)
)

print("Training images:", x_train.shape)
print("Pixel range:", x_train.min(), "to", x_train.max())


### Display an original Fashion-MNIST image

In [ ]:

plt.figure(figsize=(3, 3))
plt.imshow(x_train[0].squeeze(), cmap="gray")
plt.title("Original Fashion-MNIST Image")
plt.axis("off")
plt.show()


### Build the Generator
The Generator starts from random noise and uses transposed convolution layers to produce a 28×28 image. The final activation is **Tanh**, matching the normalized `[-1,1]` image range.

In [ ]:

def build_generator():
    model = tf.keras.Sequential([
        layers.Input(shape=(NOISE_DIM,)),
        layers.Dense(7 * 7 * 256, use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Reshape((7, 7, 256)),

        layers.Conv2DTranspose(128, kernel_size=5, strides=1,
                               padding="same", use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Conv2DTranspose(64, kernel_size=5, strides=2,
                               padding="same", use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Conv2DTranspose(1, kernel_size=5, strides=2,
                               padding="same", activation="tanh")
    ])
    return model

generator = build_generator()
generator.summary()


### Test the Generator before training
Expected output: an image that looks like random/noisy patterns because the Generator has not learned Fashion-MNIST yet.

In [ ]:

noise = tf.random.normal([16, NOISE_DIM])
generated_before_training = generator(noise, training=False)

plt.figure(figsize=(6, 6))
for i in range(16):
    plt.subplot(4, 4, i + 1)
    plt.imshow(generated_before_training[i, :, :, 0], cmap="gray")
    plt.axis("off")
plt.suptitle("Generator Before Training — Expected: Noise")
plt.tight_layout()
plt.show()


### Build the Discriminator
The Discriminator classifies real images as **1** and fake images as **0**, using a final **Sigmoid** activation.

In [ ]:

def build_discriminator():
    model = tf.keras.Sequential([
        layers.Input(shape=(28, 28, 1)),

        layers.Conv2D(64, kernel_size=5, strides=2, padding="same"),
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Conv2D(128, kernel_size=5, strides=2, padding="same"),
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Flatten(),
        layers.Dense(1, activation="sigmoid")
    ])
    return model

discriminator = build_discriminator()
discriminator.summary()


### Loss functions and optimizers
Binary Cross-Entropy is used for both networks. The Generator is trained so that fake images are classified by the Discriminator as **real (1)**.

In [ ]:

cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=False)

def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss

def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

generator_optimizer = tf.keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)
discriminator_optimizer = tf.keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)


### GAN training
The model is trained for **25 epochs**. A 4×4 grid is saved after epochs 5, 10, 15, 20, and 25. Loss values are recorded for plotting.

In [ ]:

@tf.function
def train_step(images):
    noise = tf.random.normal([BATCH_SIZE, NOISE_DIM])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)

        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)

        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    gen_gradients = gen_tape.gradient(gen_loss, generator.trainable_variables)
    disc_gradients = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(
        zip(gen_gradients, generator.trainable_variables)
    )
    discriminator_optimizer.apply_gradients(
        zip(disc_gradients, discriminator.trainable_variables)
    )

    return gen_loss, disc_loss


def save_generated_images(epoch):
    predictions = generator(hardcoded_noise, training=False)

    plt.figure(figsize=(6, 6))
    for i in range(16):
        plt.subplot(4, 4, i + 1)
        plt.imshow(predictions[i, :, :, 0], cmap="gray")
        plt.axis("off")

    plt.suptitle(f"Generated Fashion-MNIST — Epoch {epoch}")
    plt.tight_layout()
    path = f"gan_generated/epoch_{epoch:02d}.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

    return path


generator_losses = []
discriminator_losses = []
saved_epochs = []

for epoch in range(1, EPOCHS + 1):
    epoch_gen = []
    epoch_disc = []

    for batch_images in train_dataset:
        gen_loss, disc_loss = train_step(batch_images)
        epoch_gen.append(float(gen_loss))
        epoch_disc.append(float(disc_loss))

    avg_gen = np.mean(epoch_gen)
    avg_disc = np.mean(epoch_disc)

    generator_losses.append(avg_gen)
    discriminator_losses.append(avg_disc)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Generator Loss: {avg_gen:.4f} | "
        f"Discriminator Loss: {avg_disc:.4f}"
    )

    if epoch % SAVE_EVERY == 0:
        save_generated_images(epoch)
        saved_epochs.append(epoch)


### View a random saved epoch

In [ ]:

random_epoch = np.random.choice(saved_epochs)
random_path = f"gan_generated/epoch_{random_epoch:02d}.png"

img = plt.imread(random_path)
plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.axis("off")
plt.title(f"Generated Images — Random Saved Epoch {random_epoch}")
plt.show()


### Plot Generator and Discriminator losses

In [ ]:

plt.figure(figsize=(10, 5))
plt.plot(range(1, EPOCHS + 1), generator_losses, label="Generator Loss")
plt.plot(range(1, EPOCHS + 1), discriminator_losses, label="Discriminator Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("GAN Training Losses")
plt.legend()
plt.grid(True)
plt.show()


### GAN Questions — Answers

**1. How do the images change from epoch 5 to epoch 25?**  
At epoch 5, the images are mostly noisy and unclear. With more training, the Generator learns clothing shapes. By epoch 25, the images generally have more recognizable structure and detail.

**2. Can you observe instability during training?**  
Yes. GAN training can be unstable because the Generator and Discriminator compete against each other. Losses may fluctuate rather than decrease smoothly, and image quality can temporarily improve or worsen.

**3. At which epoch do cloth-like shapes become recognizable?**  
Typically around **epoch 10–15**, although the exact epoch can vary between runs.

**4. Does the GAN keep improving throughout training, or does improvement stop at some point?**  
Improvement may slow down or fluctuate after a certain point. GANs do not necessarily improve monotonically because the Generator and Discriminator continuously adapt to each other.

# Part 2 — Autoencoder and VAE using MNIST

In [ ]:

# Load MNIST
(x_train_mnist, y_train_mnist), (x_test_mnist, y_test_mnist) = tf.keras.datasets.mnist.load_data()

x_train_mnist = x_train_mnist.astype("float32") / 255.0
x_test_mnist = x_test_mnist.astype("float32") / 255.0

x_train_mnist = x_train_mnist.reshape(-1, 784)
x_test_mnist = x_test_mnist.reshape(-1, 784)

print("Training shape:", x_train_mnist.shape)
print("Test shape:", x_test_mnist.shape)


### Display MNIST before training

In [ ]:

plt.figure(figsize=(8, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train_mnist[i].reshape(28, 28), cmap="gray")
    plt.title(str(y_train_mnist[i]))
    plt.axis("off")
plt.suptitle("MNIST Before Training")
plt.tight_layout()
plt.show()


### Normal Autoencoder (AE)
The AE uses **Sigmoid** output activation and is trained for **15 epochs**.

In [ ]:

LATENT_DIM_AE = 2

ae_encoder = tf.keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation="relu"),
    layers.Dense(128, activation="relu"),
    layers.Dense(LATENT_DIM_AE, activation=None)
], name="ae_encoder")

ae_decoder = tf.keras.Sequential([
    layers.Input(shape=(LATENT_DIM_AE,)),
    layers.Dense(128, activation="relu"),
    layers.Dense(256, activation="relu"),
    layers.Dense(784, activation="sigmoid")
], name="ae_decoder")

ae = tf.keras.Sequential([ae_encoder, ae_decoder], name="autoencoder")
ae.compile(optimizer="adam", loss="mse")

ae_history = ae.fit(
    x_train_mnist, x_train_mnist,
    epochs=15,
    batch_size=256,
    validation_data=(x_test_mnist, x_test_mnist),
    verbose=1
)


### Generate the AE latent space
Colors represent different digit classes.

In [ ]:

# Use a subset for faster plotting
n_plot = 10000
ae_latent = ae_encoder.predict(x_test_mnist[:n_plot], verbose=0)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(
    ae_latent[:, 0],
    ae_latent[:, 1],
    c=y_test_mnist[:n_plot],
    cmap="tab10",
    s=5,
    alpha=0.7
)
plt.colorbar(scatter, ticks=range(10), label="Digit class")
plt.xlabel("Latent dimension 1")
plt.ylabel("Latent dimension 2")
plt.title("AE Latent Space")
plt.show()


## Variational Autoencoder (VAE)
The VAE encoder produces a mean and log variance. The latent vector is sampled using the reparameterization trick.

In [ ]:

LATENT_DIM = 2

class VAE(tf.keras.Model):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.latent_dim = latent_dim

        self.encoder_hidden = layers.Dense(256, activation="relu")
        self.encoder_hidden2 = layers.Dense(128, activation="relu")
        self.z_mean_layer = layers.Dense(latent_dim)
        self.z_log_var_layer = layers.Dense(latent_dim)

        self.decoder_hidden = layers.Dense(128, activation="relu")
        self.decoder_hidden2 = layers.Dense(256, activation="relu")
        self.decoder_output = layers.Dense(784, activation="sigmoid")

    def encode(self, x):
        h = self.encoder_hidden(x)
        h = self.encoder_hidden2(h)
        return self.z_mean_layer(h), self.z_log_var_layer(h)

    def reparameterize(self, mean, log_var):
        epsilon = tf.random.normal(shape=tf.shape(mean))
        return mean + tf.exp(0.5 * log_var) * epsilon

    def decode(self, z):
        h = self.decoder_hidden(z)
        h = self.decoder_hidden2(h)
        return self.decoder_output(h)

    def call(self, x, training=False):
        mean, log_var = self.encode(x)
        z = self.reparameterize(mean, log_var)
        reconstruction = self.decode(z)
        return reconstruction, mean, log_var


vae = VAE(LATENT_DIM)


### Define VAE loss
**Total loss = Reconstruction (MSE) + KL Divergence**.

In [ ]:

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

def vae_loss(x, reconstruction, mean, log_var):
    reconstruction_loss = tf.reduce_mean(
        tf.reduce_sum(tf.square(x - reconstruction), axis=1)
    )

    kl_loss = -0.5 * tf.reduce_mean(
        tf.reduce_sum(1 + log_var - tf.square(mean) - tf.exp(log_var), axis=1)
    )

    total_loss = reconstruction_loss + kl_loss
    return total_loss, reconstruction_loss, kl_loss


@tf.function
def vae_train_step(x):
    with tf.GradientTape() as tape:
        reconstruction, mean, log_var = vae(x, training=True)
        total, reconstruction_loss, kl_loss = vae_loss(
            x, reconstruction, mean, log_var
        )

    gradients = tape.gradient(total, vae.trainable_variables)
    optimizer.apply_gradients(zip(gradients, vae.trainable_variables))

    return total, reconstruction_loss, kl_loss


### Train VAE for 20 epochs
The notebook records **reconstruction loss, KL/divergence loss, and total loss**.

In [ ]:

vae_dataset = (
    tf.data.Dataset.from_tensor_slices(x_train_mnist)
    .shuffle(60000)
    .batch(256)
    .prefetch(tf.data.AUTOTUNE)
)

vae_total_losses = []
vae_reconstruction_losses = []
vae_kl_losses = []

for epoch in range(1, 21):
    total_values = []
    reconstruction_values = []
    kl_values = []

    for batch in vae_dataset:
        total, reconstruction_loss, kl_loss = vae_train_step(batch)
        total_values.append(float(total))
        reconstruction_values.append(float(reconstruction_loss))
        kl_values.append(float(kl_loss))

    avg_total = np.mean(total_values)
    avg_reconstruction = np.mean(reconstruction_values)
    avg_kl = np.mean(kl_values)

    vae_total_losses.append(avg_total)
    vae_reconstruction_losses.append(avg_reconstruction)
    vae_kl_losses.append(avg_kl)

    print(
        f"Epoch {epoch:02d}/20 | "
        f"Reconstruction: {avg_reconstruction:.4f} | "
        f"KL: {avg_kl:.4f} | "
        f"Total: {avg_total:.4f}"
    )


### Plot VAE losses

In [ ]:

epochs_vae = range(1, 21)

plt.figure(figsize=(10, 5))
plt.plot(epochs_vae, vae_reconstruction_losses, label="Reconstruction Loss")
plt.plot(epochs_vae, vae_kl_losses, label="KL Divergence Loss")
plt.plot(epochs_vae, vae_total_losses, label="Total Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("VAE Training Losses")
plt.legend()
plt.grid(True)
plt.show()


### Reconstruct test images
Original images are shown on top and VAE reconstructions on the bottom.

In [ ]:

test_batch = x_test_mnist[:10]
reconstructed, _, _ = vae(test_batch, training=False)
reconstructed = reconstructed.numpy()

plt.figure(figsize=(12, 4))

for i in range(10):
    plt.subplot(2, 10, i + 1)
    plt.imshow(test_batch[i].reshape(28, 28), cmap="gray")
    plt.axis("off")

    plt.subplot(2, 10, 10 + i + 1)
    plt.imshow(reconstructed[i].reshape(28, 28), cmap="gray")
    plt.axis("off")

plt.suptitle("Top: Original | Bottom: VAE Reconstruction")
plt.tight_layout()
plt.show()


### Show VAE latent space
The latent space should become relatively compact and organized. Colors indicate digit classes.

In [ ]:

mean_test, log_var_test = vae.encode(x_test_mnist[:n_plot])
vae_latent = mean_test.numpy()

plt.figure(figsize=(8, 6))
scatter = plt.scatter(
    vae_latent[:, 0],
    vae_latent[:, 1],
    c=y_test_mnist[:n_plot],
    cmap="tab10",
    s=5,
    alpha=0.7
)
plt.colorbar(scatter, ticks=range(10), label="Digit class")
plt.xlabel("Latent dimension 1")
plt.ylabel("Latent dimension 2")
plt.title("VAE Latent Space")
plt.show()


### Generate digits across the latent space
A grid of latent coordinates is decoded to observe how generated digits change continuously.

In [ ]:

grid_size = 10
z_values = np.linspace(-3, 3, grid_size)

latent_points = np.array([
    [x, y] for y in z_values[::-1] for x in z_values
], dtype=np.float32)

generated_grid = vae.decode(latent_points).numpy()

plt.figure(figsize=(10, 10))
for i in range(grid_size * grid_size):
    plt.subplot(grid_size, grid_size, i + 1)
    plt.imshow(generated_grid[i].reshape(28, 28), cmap="gray")
    plt.axis("off")

plt.suptitle("Digits Generated Across the VAE Latent Space")
plt.tight_layout()
plt.show()


### Generate images from extreme latent points

In [ ]:

extreme_points = np.array([
    [-4, -4],
    [-4,  4],
    [ 4, -4],
    [ 4,  4],
    [ 0,  0]
], dtype=np.float32)

extreme_images = vae.decode(extreme_points).numpy()

plt.figure(figsize=(10, 2))
for i in range(len(extreme_points)):
    plt.subplot(1, len(extreme_points), i + 1)
    plt.imshow(extreme_images[i].reshape(28, 28), cmap="gray")
    plt.title(str(extreme_points[i]))
    plt.axis("off")

plt.suptitle("VAE: Extreme Latent Points")
plt.tight_layout()
plt.show()


### Test a point outside the learned latent space
Points far outside the usual latent region may produce blurry, unusual, or poorly formed digits because they are outside the distribution learned during training.

In [ ]:

outside_point = np.array([[10.0, 10.0]], dtype=np.float32)
outside_image = vae.decode(outside_point).numpy()[0]

plt.figure(figsize=(3, 3))
plt.imshow(outside_image.reshape(28, 28), cmap="gray")
plt.title("Latent Point Outside the Learned Space: (10, 10)")
plt.axis("off")
plt.show()


## VAE Practical Observations

- **AE latent space:** features are encoded into a low-dimensional representation; digit classes often form distinguishable regions.
- **VAE latent space:** is encouraged to be compact and approximately Gaussian because of the KL-divergence term.
- **Reconstruction:** VAE outputs should resemble the original digits, although they can be slightly blurry.
- **Across latent space:** nearby latent points tend to produce related digit styles, demonstrating a continuous representation.
- **Extreme points:** can produce unusual or less realistic digits.
- **Outside the learned space:** very distant points can produce blurry or meaningless outputs because they are far from the latent distribution learned during training.